<a href="https://colab.research.google.com/github/petercloud23/Upload_ArquivoLocal_To_GoogleDrive/blob/main/UploadToDrive_ChooseFiles_Desktop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tkinter (Interface Desktop Local)

Esta versão abre uma janelinha para você escolher o arquivo no seu computador e fazer o upload com um clique.

Salve esse código no mesmo diretório onde está seu credentials.json

Quando rodar: Escolha o arquivo no seletor → clique em Enviar para Google Drive → veja a confirmação.

In [ ]:
import os
import tkinter as tk
from tkinter import filedialog, messagebox
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# Escopo necessário para acessar o Google Drive (apenas permissão de arquivo)
SCOPES = ['https://www.googleapis.com/auth/drive.file']

def autenticar_google():
    """Autentica no Google Drive e retorna o serviço"""
    creds = None
    # Verifica se o token.json existe (credenciais salvas de uma sessão anterior)
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    # Se as credenciais não existirem ou não forem válidas
    if not creds or not creds.valid:
        # Se as credenciais existirem, mas expiraram e há um token de atualização
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request()) # Atualiza as credenciais
        else:
            # Inicia o fluxo de autenticação do usuário (necessita de credentials.json)
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0) # Executa o servidor local para o fluxo OAuth
        # Salva as credenciais em token.json para uso futuro
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    # Constrói e retorna o objeto de serviço do Google Drive
    return build('drive', 'v3', credentials=creds)

def upload_arquivo(service, caminho_arquivo):
    """Faz upload do arquivo selecionado para o Google Drive"""
    # Verifica se o arquivo existe no caminho especificado
    if not os.path.exists(caminho_arquivo):
        messagebox.showerror("Erro", f"Arquivo não encontrado: {caminho_arquivo}")
        return

    # Obtém o nome base do arquivo a partir do caminho completo
    nome_arquivo = os.path.basename(caminho_arquivo)
    # Cria um objeto MediaFileUpload para o arquivo a ser enviado
    media = MediaFileUpload(caminho_arquivo, resumable=True)
    # Define os metadados do arquivo (apenas o nome por enquanto)
    file_metadata = {'name': nome_arquivo}

    # Faz a requisição para criar o arquivo no Google Drive
    file = service.files().create(body=file_metadata, media_body=media, fields='id').execute()
    # Exibe uma mensagem de sucesso com o nome e ID do arquivo enviado
    messagebox.showinfo("Sucesso", f"Arquivo enviado!\nNome: {nome_arquivo}\nID: {file.get('id')}")

def escolher_arquivo():
    """Abre o seletor de arquivo para o usuário escolher um arquivo"""
    # Abre a janela de diálogo para selecionar um arquivo
    caminho = filedialog.askopenfilename(title="Escolha um arquivo para enviar")
    # Se um arquivo for selecionado (o caminho não é vazio)
    if caminho:
        # Limpa o conteúdo atual do campo de entrada
        entry_caminho.delete(0, tk.END)
        # Insere o caminho do arquivo selecionado no campo de entrada
        entry_caminho.insert(0, caminho)

def enviar_para_drive():
    """Função chamada pelo botão para iniciar o processo de upload"""
    # Obtém o caminho do arquivo do campo de entrada
    caminho = entry_caminho.get()
    # Verifica se um caminho de arquivo foi inserido
    if not caminho:
        messagebox.showwarning("Atenção", "Por favor, selecione um arquivo primeiro.")
        return
    try:
        # Autentica no Google Drive
        service = autenticar_google()
        # Faz o upload do arquivo usando o serviço autenticado e o caminho do arquivo
        upload_arquivo(service, caminho)
    except Exception as e:
        # Em caso de erro, exibe uma mensagem de erro
        messagebox.showerror("Erro", f"Ocorreu um erro:\n{e}")

# --- Interface Tkinter ---
# Cria a janela principal do aplicativo
janela = tk.Tk()
# Define o título da janela
janela.title("Upload para Google Drive")
# Define o tamanho inicial da janela
janela.geometry("500x200")

# Cria e empacota um Label para instrução
tk.Label(janela, text="Selecione um arquivo para enviar:", font=("Arial", 12)).pack(pady=10)

# Cria um Frame para agrupar o campo de entrada e o botão "Procurar"
frame = tk.Frame(janela)
frame.pack(pady=5)

# Cria o campo de entrada para exibir o caminho do arquivo
entry_caminho = tk.Entry(frame, width=50)
entry_caminho.pack(side=tk.LEFT, padx=5)

# Cria o botão "Procurar" que chama a função escolher_arquivo
btn_browse = tk.Button(frame, text="Procurar", command=escolher_arquivo)
btn_browse.pack(side=tk.LEFT)

# Cria o botão "Enviar para Google Drive" que chama a função enviar_para_drive
btn_upload = tk.Button(janela, text="Enviar para Google Drive", command=enviar_para_drive, bg="green", fg="white", font=("Arial", 12))
btn_upload.pack(pady=20)

# Inicia o loop principal da interface gráfica
janela.mainloop()